# Module 5 — Inverse Kinematics
## Unit 6 — Singularities and Solution Selection
### Lesson 6.2 — Joint Limits and Feasible Solutions

*Physical AI Curriculum · hands-on notebook. Run **Kernel → Restart & Run All**.*


## Demonstration (§7)
Filter the two solutions against joint limits.


In [ ]:
import numpy as np
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt

def fk_two_link(t1, t2, L1, L2):
    return np.array([L1*np.cos(t1)+L2*np.cos(t1+t2), L1*np.sin(t1)+L2*np.sin(t1+t2)])

def jacobian_2link(t1, t2, L1, L2):
    s1,s12=np.sin(t1),np.sin(t1+t2); c1,c12=np.cos(t1),np.cos(t1+t2)
    return np.array([[-L1*s1-L2*s12, -L2*s12],[L1*c1+L2*c12, L2*c12]])

def ik_2link_closed(x, y, L1, L2):
    c2=(x*x+y*y-L1*L1-L2*L2)/(2*L1*L2)
    if c2<-1-1e-9 or c2>1+1e-9: return []
    c2=max(-1.0,min(1.0,c2)); sols=[]
    for sign in (+1.0,-1.0):
        s2=sign*np.sqrt(max(0.0,1-c2*c2)); t2=np.arctan2(s2,c2)
        t1=np.arctan2(y,x)-np.arctan2(L2*np.sin(t2),L1+L2*np.cos(t2))
        sols.append((t1,t2))
        if abs(s2)<1e-6: break
    return sols

L1,L2=0.4,0.3
def norm180(a):  # wrap degrees to (-180,180]
    return (a+180)%360-180
def feasible_solutions(sols, limits):
    out=[]
    for (t1,t2) in sols:
        d1,d2=norm180(np.degrees(t1)),norm180(np.degrees(t2))
        (a1,b1),(a2,b2)=limits
        if a1<=d1<=b1 and a2<=d2<=b2: out.append((t1,t2))
    return out
sols=ik_2link_closed(0.5,0.0,L1,L2)
limits=[(-45,45),(0,150)]
feas=feasible_solutions(sols,limits)
print("all:",[tuple(np.round(np.degrees(s),2)) for s in sols])
print("feasible:",[tuple(np.round(np.degrees(s),2)) for s in feas])


## Coding Exercise (§8)
Worked example keeps elbow-down; build a none-feasible case.


In [ ]:
# YOUR CODE HERE
sols=ik_2link_closed(0.5,0.0,L1,L2)
feas=feasible_solutions(sols,[(-45,45),(0,150)])
assert len(feas)==1 and feas[0][1]>0          # only elbow-down (theta2=+90)
none=feasible_solutions(sols,[(-5,5),(0,10)])  # excludes both
assert len(none)==0
print("joint-limit filtering OK")


## Check your work


In [ ]:
feas=feasible_solutions(ik_2link_closed(0.5,0.0,L1,L2),[(-45,45),(0,150)])
assert abs(np.degrees(feas[0][1])-90)<1e-6
print("All checks passed.")
